# Phase 2: Data Understanding & Exploratory Data Analysis
## Customer Segmentation & Retention Analysis
Olist E-commerce Dataset

### Objectives:
- Understand structure of all datasets
- Examine data types and missing values
- Identify key relationships
- Prepare dataset for feature engineering


In [ ]:
# =============================
# Imports
# =============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# =============================
# Load Datasets
# =============================

import os
import pandas as pd

# Using relative path 
DATA_PATH = "../dataset"

# Debug checking
print("Files inside dataset folder:")
print(os.listdir(DATA_PATH))

# Loading files safely
customers = pd.read_csv(os.path.join(DATA_PATH, "olist_customers_dataset.csv"))
orders = pd.read_csv(os.path.join(DATA_PATH, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(DATA_PATH, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(DATA_PATH, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(DATA_PATH, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(DATA_PATH, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(DATA_PATH, "olist_sellers_dataset.csv"))

print("✅ All datasets loaded successfully!")


In [ ]:
# =============================
# Dataset Overview Function
# =============================

def dataset_overview(df, name):
    print("="*50)
    print(f"Dataset: {name}")
    print("="*50)
    print("Shape:", df.shape)
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum().sort_values(ascending=False))
    print("\nSample Data:")
    display(df.head())

# Call function 
dataset_overview(customers, "Customers")
dataset_overview(orders, "Orders")
dataset_overview(order_items, "Order Items")
dataset_overview(order_payments, "Order payments")
dataset_overview(order_reviews, "Order reviews")
dataset_overview(products, "Products")
dataset_overview(sellers,"Sellers")

# Phase 2: Data Understanding & Exploratory Data Analysis (EDA)

In this phase, we perform a structured exploratory analysis to:
- Validate data quality
- Understand dataset relationships
- Compute key business metrics
- Prepare a consolidated dataset for segmentation and retention analysis


In [ ]:
# ============================================
# 1. Imports & Display Settings
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

sns.set(style="whitegrid")


In [ ]:
# ============================================
# 2. Data Quality Checks
# ============================================

def check_uniqueness(df, column, name):
    print(f"{name} - Unique {column}: {df[column].nunique()}")
    print(f"{name} - Total Rows: {df.shape[0]}")
    print("-"*40)

check_uniqueness(customers, 'customer_id', 'Customers')
check_uniqueness(orders, 'order_id', 'Orders')

print("Duplicate rows check:")
print("Customers:", customers.duplicated().sum())
print("Orders:", orders.duplicated().sum())


In [ ]:
# ============================================
# 3. Datetime Conversion
# ============================================

orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

print("Date Range:")
print("From:", orders['order_purchase_timestamp'].min())
print("To:", orders['order_purchase_timestamp'].max())


In [ ]:
# ============================================
# 4. Core Business KPIs
# ============================================

total_customers = customers['customer_id'].nunique()
total_orders = orders['order_id'].nunique()
total_revenue = order_payments['payment_value'].sum()
avg_review_score = order_reviews['review_score'].mean()

print(f"Total Customers: {total_customers:,}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Average Review Score: {avg_review_score:.2f}")

In [ ]:
# ============================================
# 5. Creating Master Order Dataset
# ============================================

master_df = (
    orders
    .merge(order_payments, on='order_id', how='left')
    .merge(order_reviews, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
)

print("Master Dataset Shape:", master_df.shape)
master_df.head()

In [ ]:
# ============================================
# 6. Revenue Distribution
# ============================================

master_df['payment_value'].describe()

In [ ]:
plt.figure()
master_df['payment_value'].hist(bins=50)
plt.title("Distribution of Order Payment Value")
plt.xlabel("Payment Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# ============================================
# 7. Orders Per Customer
# ============================================

orders_per_customer = (
    master_df.groupby('customer_id')['order_id']
    .nunique()
    .sort_values(ascending=False)
)

orders_per_customer.describe()

In [ ]:
plt.figure()
orders_per_customer.hist(bins=30)
plt.title("Orders per Customer Distribution")
plt.xlabel("Number of Orders")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# ============================================
# 8. Time-Based Trend Analysis
# ============================================

# Ensure datetime format (safe check)
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

# Extract year and month
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

In [ ]:
# ============================================
# Yearly Order Trend
# ============================================

yearly_orders = (
    orders.groupby('order_year')['order_id']
    .nunique()
    .sort_index()
)

print("Yearly Order Count:")
display(yearly_orders)

In [ ]:
plt.figure()
yearly_orders.plot(marker='o')
plt.title("Yearly Order Trend")
plt.xlabel("Year")
plt.ylabel("Number of Orders")
plt.show()

In [ ]:
# ============================================
# Monthly Order Trend
# ============================================

monthly_orders = (
    orders.groupby('order_month')['order_id']
    .nunique()
    .sort_index()
)

print("Monthly Order Count:")
display(monthly_orders.head())
monthly_orders.index = monthly_orders.index.to_timestamp()

In [ ]:
plt.figure()
monthly_orders.plot()
plt.title("Monthly Order Trend")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.show()

### Observations

- Order volume shows a clear growth pattern over time.
- Certain months indicate higher peaks, suggesting possible seasonality.
- Growth rate fluctuations suggest potential promotional or demand cycles.

These insights are important for customer retention modeling and future cohort analysis.

In [ ]:
# ============================================
# Ensure processed directory exists
# ============================================

import os

os.makedirs("../data/processed", exist_ok=True)

# Save master dataset
master_df.to_csv("../data/processed/master_dataset.csv", index=False)

print("Master dataset saved successfully.")